# Notebook 48 — Zscore Mixing Nonlinearity

**Question from nb47:** The centroid-midpoint model (classify 0.5×ctrd_i + 0.5×ctrd_j) achieves only 45.3% accuracy for the composition table — it over-predicts oscillator and misses declining_osc. The proposed explanation: the `zscore(0.5×A + 0.5×B)` operation is nonlinear. For uncorrelated unit-variance signals, it compresses skewness (~1/√2) and kurtosis (~1/2) and amplifies slope/baseline_delta (~√2) relative to the centroid midpoint.

**Approach:** For each of the 64 (i,j) pairs, compute the *mean actual feature vector* from 500 mixed-signal samples and compare it to the centroid midpoint. Directly measure which features deviate and by how much.

---

## Pre-run predictions

**F151:** Skewness and kurtosis are compressed toward zero: actual ≈ midpoint × 1/√2 (skewness) and midpoint × 0.5 (kurtosis). Pearson ρ(actual, midpoint) > 0.90 for both — the compression is proportional, not random.

**F152:** Slope and baseline_delta are amplified: actual ≈ midpoint × √2. Pearson ρ > 0.90. Lag1 and ZC are approximately linear (ρ > 0.90, ratio ≈ 1.0).

**F153:** Using actual mean features improves composition prediction accuracy from 45.3% (nb47 centroid-midpoint) to >65%.

**F154:** For all 24 pairs that empirically produce declining_osc, the actual mean features are closer to the declining_osc centroid than the centroid midpoint — the zscore nonlinearity consistently pulls mixed signals toward the declining_osc basin.

In [1]:
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from scipy.spatial.distance import pdist, squareform
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import time, os, sys
os.makedirs('../artifacts', exist_ok=True)
sys.path.insert(0, '..')

SIGNED_COLS = ['skewness', 'kurtosis', 'lag1_autocorr', 'zero_crossings', 'slope', 'baseline_delta']
SEQ_LEN = 64; SEED = 42; t64 = np.linspace(0, 1, SEQ_LEN)

def zscore(s):
    s = np.asarray(s, dtype=float); std = s.std()
    return (s - s.mean()) / std if std > 1e-8 else s * 0.0

def baseline_delta_fn(s, frac=0.10):
    k = max(1, int(len(s) * frac))
    return float(np.mean(s[-k:]) - np.mean(s[:k]))

def extract_6f(s):
    arr = np.asarray(s, dtype=float); t = np.arange(len(arr))
    lag1 = float(np.corrcoef(arr[:-1], arr[1:])[0, 1]) if len(arr) > 2 else 0.0
    return {
        'skewness':       float(stats.skew(arr)),
        'kurtosis':       float(stats.kurtosis(arr)),
        'lag1_autocorr':  lag1,
        'zero_crossings': float(np.sum(np.diff(np.sign(arr)) != 0) / len(arr)),
        'slope':          float(stats.linregress(t, arr).slope),
        'baseline_delta': baseline_delta_fn(arr),
    }

GENERATORS = {
    'burst':              lambda r: zscore(np.exp(-(t64-r.uniform(.15,.50))**2/(2*r.uniform(.05,.15)**2))+r.normal(0,.05,SEQ_LEN)),
    'oscillator':         lambda r: zscore(np.sin(2*np.pi*r.uniform(1.5,4.5)*t64+r.uniform(0,np.pi))+r.normal(0,.05,SEQ_LEN)),
    'seasonal':           lambda r: zscore(np.sin(2*np.pi*r.uniform(3,6)*t64)+.25*np.sin(4*np.pi*r.uniform(3,6)*t64)+r.normal(0,.04,SEQ_LEN)),
    'trend':              lambda r: zscore(t64+r.uniform(.05,.30)*t64**2+r.normal(0,.02,SEQ_LEN)),
    'integrated_trend':   lambda r: zscore(np.cumsum(np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
    'irregular_osc':      lambda r: zscore((np.sin(2*np.pi*r.uniform(2,5)*t64)*(1+r.uniform(.3,.8,SEQ_LEN))+r.normal(0,.3,SEQ_LEN))*1.4),
    'declining_osc':      lambda r: zscore(np.linspace(r.uniform(.9,1.2),r.uniform(.35,.65),SEQ_LEN)*np.sin(2*np.pi*r.uniform(2.5,5.5)*t64)+np.linspace(0,r.uniform(-.8,-.4),SEQ_LEN)+r.normal(0,.05,SEQ_LEN)),
    'declining_monotonic':lambda r: zscore(np.cumsum(-np.ones(SEQ_LEN)*r.uniform(.015,.035)+r.normal(0,.003,SEQ_LEN))),
}

CLASSES = list(GENERATORS.keys())
N_CLASSES = len(CLASSES)
CLS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
ABBREV = {
    'burst': 'BUR', 'oscillator': 'OSC', 'seasonal': 'SEA',
    'trend': 'TRE', 'integrated_trend': 'INT', 'irregular_osc': 'IRR',
    'declining_osc': 'DCO', 'declining_monotonic': 'DCM',
}

recs = []
for cls, gen in GENERATORS.items():
    for i in range(200):
        r = np.random.default_rng(SEED + CLASSES.index(cls)*1000 + i)
        f = extract_6f(gen(r)); f['class'] = cls; recs.append(f)
df_fit = pd.DataFrame(recs)
sc = StandardScaler()
X_fit = sc.fit_transform(df_fit[SIGNED_COLS].values)
ctrds = {c: X_fit[df_fit['class']==c].mean(axis=0) for c in GENERATORS}
ctrd_arr = np.array([ctrds[c] for c in CLASSES])  # (8, 6)

def classify_scaled(x):
    dists = {c: float(np.linalg.norm(x - ctrds[c])) for c in CLASSES}
    return min(dists, key=dists.get), min(dists.values())

# Re-derive composition table (same seeds as nb46/47)
table_name = [['' for _ in range(N_CLASSES)] for _ in range(N_CLASSES)]
N_SAMPLES = 500
print('Re-deriving composition table ...')
t0 = time.time()
for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        results = [classify_scaled(
            sc.transform([[v for v in extract_6f(zscore(
                0.5*GENERATORS[cls_a](np.random.default_rng(1000+i*5000+k)) +
                0.5*GENERATORS[cls_b](np.random.default_rng(2000+j*5000+k))
            )).values()]])[0])[0]
            for k in range(N_SAMPLES)]
        table_name[i][j] = Counter(results).most_common(1)[0][0]
print(f'Done in {time.time()-t0:.1f}s')

Re-deriving composition table ...


Done in 15.5s


In [2]:
# ---- Part A: Compute actual mean feature vectors for all 64 pairs ----

print('=== Part A: Mean actual features vs centroid midpoints ===')
print(f'(N={N_SAMPLES} samples per pair, 64 pairs ...)')

t0 = time.time()
mean_actual  = np.zeros((N_CLASSES, N_CLASSES, 6))  # (8, 8, 6) scaled
std_actual   = np.zeros((N_CLASSES, N_CLASSES, 6))
midpoints    = np.zeros((N_CLASSES, N_CLASSES, 6))

for i, cls_a in enumerate(CLASSES):
    for j, cls_b in enumerate(CLASSES):
        feat_vecs = []
        for k in range(N_SAMPLES):
            r_a = np.random.default_rng(1000 + i*5000 + k)
            r_b = np.random.default_rng(2000 + j*5000 + k)
            mixed = zscore(0.5*GENERATORS[cls_a](r_a) + 0.5*GENERATORS[cls_b](r_b))
            fp = extract_6f(mixed)
            x_scaled = sc.transform([[fp[c] for c in SIGNED_COLS]])[0]
            feat_vecs.append(x_scaled)
        fv = np.array(feat_vecs)  # (N, 6)
        mean_actual[i, j] = fv.mean(axis=0)
        std_actual[i, j]  = fv.std(axis=0)
        midpoints[i, j]   = 0.5 * (ctrd_arr[i] + ctrd_arr[j])

deviations = mean_actual - midpoints  # (8, 8, 6)  actual - midpoint
print(f'Done in {time.time()-t0:.1f}s')
print()

# Summary: mean deviation per feature
print(f'{"Feature":20s}  {"Mean dev":>10s}  {"MAE dev":>10s}  {"Dev/Midpt":>12s}')
print('-'*60)
for k, feat in enumerate(SIGNED_COLS):
    devs   = deviations[:, :, k].flatten()
    midpts = midpoints[:, :, k].flatten()
    nonzero_mask = np.abs(midpts) > 0.05
    ratio = np.median(devs[nonzero_mask] / midpts[nonzero_mask]) if nonzero_mask.sum() > 0 else 0
    print(f'{feat:20s}  {devs.mean():+10.4f}  {np.abs(devs).mean():10.4f}  {ratio:+12.4f}')

=== Part A: Mean actual features vs centroid midpoints ===
(N=500 samples per pair, 64 pairs ...)


Done in 15.3s

Feature                 Mean dev     MAE dev     Dev/Midpt
------------------------------------------------------------
skewness                 -0.2044      0.4925       -0.4938
kurtosis                 +0.6262      0.7229       -2.3754
lag1_autocorr            -0.5247      0.5995       -0.0369
zero_crossings           +0.7431      0.7613       +0.0959
slope                    +0.0059      0.2036       +0.4346
baseline_delta           -0.0098      0.2038       +0.4130


In [3]:
# ---- Part B: Per-feature linearity analysis ----

print('=== Part B: Actual vs Midpoint — per-feature Pearson ρ and compression/amplification ratio ===')
print()
print(f'{"Feature":20s}  {"Pearson ρ":>10s}  {"p-value":>10s}  {"Slope (actual/midpt)":>22s}  {"Predicted ratio":>16s}')
print('-'*85)

predicted_ratios = {
    'skewness':       1/np.sqrt(2),   # ~0.707
    'kurtosis':       0.5,            # compressed by 2
    'lag1_autocorr':  1.0,            # linear
    'zero_crossings': 1.0,            # approx linear
    'slope':          np.sqrt(2),     # ~1.414 amplified
    'baseline_delta': np.sqrt(2),     # ~1.414 amplified
}

feature_stats = {}
for k, feat in enumerate(SIGNED_COLS):
    actual_vals = mean_actual[:, :, k].flatten()
    midpt_vals  = midpoints[:, :, k].flatten()
    rho, pval   = pearsonr(actual_vals, midpt_vals)
    # Slope of actual vs midpt via linear regression (no intercept)
    slope_nointerc = np.dot(actual_vals, midpt_vals) / (np.dot(midpt_vals, midpt_vals) + 1e-12)
    pred_ratio = predicted_ratios[feat]
    feature_stats[feat] = {'rho': rho, 'pval': pval, 'slope': slope_nointerc, 'pred': pred_ratio}
    print(f'{feat:20s}  {rho:+10.4f}  {pval:10.4f}  {slope_nointerc:22.4f}  {pred_ratio:16.4f}')

print()
print('F151 check: skewness compressed (slope < 1.0)?', feature_stats['skewness']['slope'] < 1.0)
print('F151 check: kurtosis compressed (slope < 1.0)?', feature_stats['kurtosis']['slope'] < 1.0)
print('F152 check: slope amplified (slope > 1.0)?',     feature_stats['slope']['slope'] > 1.0)
print('F152 check: BD amplified (slope > 1.0)?',        feature_stats['baseline_delta']['slope'] > 1.0)

=== Part B: Actual vs Midpoint — per-feature Pearson ρ and compression/amplification ratio ===

Feature                Pearson ρ     p-value    Slope (actual/midpt)   Predicted ratio
-------------------------------------------------------------------------------------
skewness                 +0.2388      0.0573                  0.2112            0.7071
kurtosis                 +0.3403      0.0059                  0.2574            0.5000
lag1_autocorr            -0.0395      0.7565                 -0.1291            1.0000
zero_crossings           +0.2360      0.0604                  0.4153            1.0000
slope                    +0.9835      0.0000                  1.2536            1.4142
baseline_delta           +0.9818      0.0000                  1.2656            1.4142

F151 check: skewness compressed (slope < 1.0)? True
F151 check: kurtosis compressed (slope < 1.0)? True
F152 check: slope amplified (slope > 1.0)? True
F152 check: BD amplified (slope > 1.0)? True


In [4]:
# ---- Part C: Prediction accuracy with actual mean features ----

print('=== Part C: Prediction accuracy — centroid midpoint vs actual mean features ===')
print()

correct_mid = 0
correct_act = 0
changes = []  # pairs where prediction changes

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        emp = table_name[i][j]
        pred_mid, _ = classify_scaled(midpoints[i, j])
        pred_act, _ = classify_scaled(mean_actual[i, j])
        if pred_mid == emp:
            correct_mid += 1
        if pred_act == emp:
            correct_act += 1
        if pred_mid != pred_act:
            changes.append((CLASSES[i], CLASSES[j], emp, pred_mid, pred_act))

print(f'Centroid-midpoint accuracy:    {correct_mid}/64 = {correct_mid/64:.1%}  (nb47)')
print(f'Actual mean-feature accuracy:  {correct_act}/64 = {correct_act/64:.1%}  (nb48)')
print(f'F153 check: >65%? {correct_act/64 > 0.65}')
print()

print(f'Pairs where prediction changes ({len(changes)} pairs):')
print(f'{"Pair":35s}  {"Empirical":20s}  {"Midpt pred":20s}  {"Actual pred":20s}  {"Improvement"}')
print('-'*115)
n_improved = 0
for cls_a, cls_b, emp, pred_mid, pred_act in changes:
    improvement = '✓' if pred_act == emp else ('✗' if pred_mid == emp else '~')
    if pred_act == emp:
        n_improved += 1
    print(f'  ({cls_a[:8]:8s}, {cls_b[:8]:8s})  {emp:20s}  {pred_mid:20s}  {pred_act:20s}  {improvement}')

n_degraded = sum(1 for _, _, e, pm, pa in changes if pm == e and pa != e)
print()
print(f'Changed to correct:   {n_improved}')
print(f'Changed to incorrect: {n_degraded}')

=== Part C: Prediction accuracy — centroid midpoint vs actual mean features ===

Centroid-midpoint accuracy:    29/64 = 45.3%  (nb47)
Actual mean-feature accuracy:  62/64 = 96.9%  (nb48)
F153 check: >65%? True

Pairs where prediction changes (33 pairs):
Pair                                 Empirical             Midpt pred            Actual pred           Improvement
-------------------------------------------------------------------------------------------------------------------
  (burst   , trend   )  integrated_trend      burst                 integrated_trend      ✓
  (burst   , integrat)  integrated_trend      trend                 integrated_trend      ✓
  (burst   , declinin)  declining_osc         burst                 declining_osc         ✓
  (burst   , declinin)  declining_monotonic   burst                 declining_monotonic   ✓
  (oscillat, seasonal)  declining_osc         seasonal              declining_osc         ✓
  (oscillat, trend   )  trend                 oscillato

In [5]:
# ---- Part D: Does deviation point toward declining_osc? ----

print('=== Part D: Does the zscore nonlinearity pull mixed features toward declining_osc? ===')
print()

dco_idx  = CLASSES.index('declining_osc')
dco_ctrd = ctrd_arr[dco_idx]
osc_idx  = CLASSES.index('oscillator')
osc_ctrd = ctrd_arr[osc_idx]

# For each of the 64 pairs: does the deviation move the point closer to DCO vs OSC?
mid_to_dco  = np.array([[np.linalg.norm(midpoints[i,j] - dco_ctrd) for j in range(N_CLASSES)] for i in range(N_CLASSES)])
act_to_dco  = np.array([[np.linalg.norm(mean_actual[i,j] - dco_ctrd) for j in range(N_CLASSES)] for i in range(N_CLASSES)])
mid_to_osc  = np.array([[np.linalg.norm(midpoints[i,j] - osc_ctrd)  for j in range(N_CLASSES)] for i in range(N_CLASSES)])
act_to_osc  = np.array([[np.linalg.norm(mean_actual[i,j] - osc_ctrd)  for j in range(N_CLASSES)] for i in range(N_CLASSES)])

# Closer to DCO after nonlinearity?
closer_to_dco = (act_to_dco < mid_to_dco).sum()
# Farther from OSC after nonlinearity?
farther_from_osc = (act_to_osc > mid_to_osc).sum()

print(f'All 64 pairs: actual closer to DCO centroid than midpoint: {closer_to_dco}/64 = {closer_to_dco/64:.1%}')
print(f'All 64 pairs: actual farther from OSC centroid than midpoint: {farther_from_osc}/64 = {farther_from_osc/64:.1%}')
print()

# For DCO-output pairs specifically
dco_pairs = [(i, j) for i in range(N_CLASSES) for j in range(N_CLASSES) if table_name[i][j] == 'declining_osc']
dco_closer = sum(1 for (i,j) in dco_pairs if act_to_dco[i,j] < mid_to_dco[i,j])
print(f'DCO-output pairs ({len(dco_pairs)}): actual closer to DCO: {dco_closer}/{len(dco_pairs)} = {dco_closer/len(dco_pairs):.1%}')
print(f'F154 check: = 100%? {dco_closer == len(dco_pairs)}')
print()

# Quantify: mean distance reduction to DCO for DCO pairs
dist_reductions = [mid_to_dco[i,j] - act_to_dco[i,j] for (i,j) in dco_pairs]
print(f'DCO pairs — distance reduction to DCO centroid (midpt→actual):')
print(f'  mean: {np.mean(dist_reductions):.4f}  median: {np.median(dist_reductions):.4f}  max: {max(dist_reductions):.4f}')
print()

# The key question: which features drive the deviation toward DCO?
# For all pairs where the prediction changes from OSC (midpt) to DCO (actual):
osc_to_dco = [(cls_a, cls_b, emp, pm, pa) for (cls_a, cls_b, emp, pm, pa) in changes
               if pm == 'oscillator' and pa == 'declining_osc']
print(f'Pairs where prediction changes from oscillator→declining_osc ({len(osc_to_dco)}): ')
for cls_a, cls_b, emp, pm, pa in osc_to_dco:
    i, j = CLASSES.index(cls_a), CLASSES.index(cls_b)
    dev = deviations[i, j]
    print(f'  ({cls_a[:8]}, {cls_b[:8]}) emp={emp[:6]:6s}: dev=[' +
          ', '.join(f'{v:+.3f}' for v in dev) + ']')
    print(f'    skew_dev={dev[0]:+.3f} kurt_dev={dev[1]:+.3f} slope_dev={dev[4]:+.3f}')

=== Part D: Does the zscore nonlinearity pull mixed features toward declining_osc? ===

All 64 pairs: actual closer to DCO centroid than midpoint: 24/64 = 37.5%
All 64 pairs: actual farther from OSC centroid than midpoint: 54/64 = 84.4%

DCO-output pairs (25): actual closer to DCO: 16/25 = 64.0%
F154 check: = 100%? False

DCO pairs — distance reduction to DCO centroid (midpt→actual):
  mean: 0.0065  median: 0.0455  max: 0.4322

Pairs where prediction changes from oscillator→declining_osc (4): 
  (oscillat, declinin) emp=declin: dev=[-0.053, +0.788, +0.035, +0.092, -0.147, -0.249]
    skew_dev=-0.053 kurt_dev=+0.788 slope_dev=-0.147
  (oscillat, declinin) emp=declin: dev=[+0.083, +0.998, -0.004, +0.501, -0.306, -0.316]
    skew_dev=+0.083 kurt_dev=+0.998 slope_dev=-0.306
  (declinin, oscillat) emp=declin: dev=[-0.073, +0.802, +0.062, +0.058, -0.147, -0.240]
    skew_dev=-0.073 kurt_dev=+0.802 slope_dev=-0.147
  (declinin, oscillat) emp=declin: dev=[+0.090, +0.961, +0.007, +0.495, -0.306

In [6]:
# ---- Visualisation ----

CLASS_COLORS = {
    'burst': '#F44336', 'oscillator': '#2196F3', 'seasonal': '#FF9800',
    'trend': '#795548', 'integrated_trend': '#607D8B', 'irregular_osc': '#E91E63',
    'declining_osc': '#9C27B0', 'declining_monotonic': '#009688',
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('nb48 — Zscore Mixing Nonlinearity', fontsize=13, fontweight='bold')

# Panels 1-6: per-feature actual vs midpoint scatter
theory_slopes = [1/np.sqrt(2), 0.5, 1.0, 1.0, np.sqrt(2), np.sqrt(2)]

for k, (feat, ax) in enumerate(zip(SIGNED_COLS, axes.flat)):
    actual_vals = mean_actual[:, :, k].flatten()
    midpt_vals  = midpoints[:, :, k].flatten()
    emp_colors  = [CLASS_COLORS[table_name[i][j]]
                   for i in range(N_CLASSES) for j in range(N_CLASSES)]
    
    ax.scatter(midpt_vals, actual_vals, c=emp_colors, alpha=0.7, s=30, zorder=3)
    
    # Identity line
    mn = min(midpt_vals.min(), actual_vals.min())
    mx = max(midpt_vals.max(), actual_vals.max())
    ax.plot([mn, mx], [mn, mx], 'k--', lw=1, alpha=0.4, label='y=x (linear)')
    
    # Theory prediction line
    ts = theory_slopes[k]
    ax.plot([mn, mx], [ts*mn, ts*mx], 'r-', lw=1.5, alpha=0.7, label=f'Theory slope={ts:.3f}')
    
    rho = feature_stats[feat]['rho']
    sl  = feature_stats[feat]['slope']
    ax.set_xlabel(f'Midpoint {feat[:6]}', fontsize=9)
    ax.set_ylabel(f'Actual {feat[:6]}', fontsize=9)
    ax.set_title(f'{feat}\nρ={rho:.3f}  slope={sl:.3f}', fontsize=9)
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig('../artifacts/nb48_zscore_nonlinearity.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure saved.')

# PCA: class centroids, centroid midpoints, and actual mean features
fig2, ax2 = plt.subplots(1, 1, figsize=(10, 8))
all_pts = np.vstack([
    ctrd_arr,                          # 8 centroids
    midpoints.reshape(-1, 6),          # 64 midpoints
    mean_actual.reshape(-1, 6),        # 64 actual means
])
pca2 = PCA(n_components=2)
all_2d = pca2.fit_transform(all_pts)

ctrd_2d  = all_2d[:8]
mid_2d   = all_2d[8:72].reshape(N_CLASSES, N_CLASSES, 2)
act_2d   = all_2d[72:].reshape(N_CLASSES, N_CLASSES, 2)

# Draw arrows: midpoint → actual
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        ec = CLASS_COLORS[table_name[i][j]]
        ax2.annotate('', xy=act_2d[i,j], xytext=mid_2d[i,j],
                     arrowprops=dict(arrowstyle='->', color=ec, lw=0.8, alpha=0.6))

# Draw class centroids
for i, c in enumerate(CLASSES):
    ax2.scatter(ctrd_2d[i, 0], ctrd_2d[i, 1], s=300, color=CLASS_COLORS[c],
               zorder=6, edgecolors='black', linewidths=1.5)
    ax2.annotate(ABBREV[c], (ctrd_2d[i, 0], ctrd_2d[i, 1]), fontsize=10,
                fontweight='bold', ha='center', va='center', color='white',
                zorder=7)

ax2.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.0f}%)')
ax2.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.0f}%)')
ax2.set_title('PCA: centroid midpoints → actual mean features\n(arrow = nonlinear shift, colored by empirical T[i,j])')

# Legend
from matplotlib.patches import Patch
legend_patches = [Patch(color=CLASS_COLORS[c], label=ABBREV[c]) for c in CLASSES]
ax2.legend(handles=legend_patches, fontsize=9, loc='best', title='T[i,j]')

plt.tight_layout()
plt.savefig('../artifacts/nb48_pca_shift.png', dpi=120, bbox_inches='tight')
plt.show()
print('PCA figure saved.')

Figure saved.


PCA figure saved.


---
## Findings — Notebook 48

### F151 — Skewness/kurtosis compression under zscore mixing (PARTIAL)

**Prediction:** actual ≈ midpoint/√2 for skewness (slope ≈ 0.707), midpoint/2 for kurtosis (slope ≈ 0.5). Pearson ρ > 0.90.

**Result (Part B):**
- skewness: ρ = +0.239, slope (no-intercept) = **0.21** — compressed (<1.0) ✓ but **far more aggressive** than the predicted 0.707; correlation weak.
- kurtosis: ρ = +0.340, slope = **0.26** — compressed (<1.0) ✓ but more aggressive than the predicted 0.50; correlation weak.

**Verdict:** Direction confirmed (compression). Magnitude exceeds the simple "uncorrelated unit-variance" theory — the zscore-after-mix collapses higher-moment information almost entirely. ρ < 0.5 indicates the mapping is not a clean rescaling — higher-order moments of the mixture are dominated by the *interaction* of A and B, not their individual values.

---

### F152 — Slope/BD amplification; lag1/ZC linearity (PARTIAL)

**Prediction:** slope/BD amplified (slope ≈ √2 ≈ 1.414), lag1/ZC linear (slope ≈ 1.0). Pearson ρ > 0.90 for all four.

**Result (Part B):**
- slope: ρ = **+0.984**, slope-no-intercept = **1.254** — amplified ✓, near √2 (slight under-amplification).
- baseline_delta: ρ = **+0.982**, slope = **1.266** — amplified ✓, near √2.
- lag1_autocorr: ρ = **−0.040**, slope = **−0.129** — **NOT linear**. The mixed lag-1 is essentially decorrelated from the midpoint of the constituent lag-1s.
- zero_crossings: ρ = +0.236, slope = **0.415** — **NOT linear**. Compressed and weakly correlated.

**Verdict:** First-moment-like features (slope, baseline_delta) follow theory cleanly. Second-order shape features (lag1, ZC) violate theory: they are nonlinear functionals of the *mixed* signal, not interpolations of their constituents.

---

### F153 — Accuracy improvement with actual mean features (CONFIRMED, exceeds prediction)

**Prediction:** >65% accuracy with actual mean features (vs 45.3% with centroid midpoints).

**Result (Part C):** **62/64 = 96.9%** (vs 45.3% with midpoints). All 33 changed predictions move from incorrect → correct (none regress).

**Verdict:** Massive overshoot. The empirical composition table is almost entirely determined by the *mean of the actual mixed-and-zscored feature distribution*, not by centroid interpolation. The zscore-after-mixing nonlinearity is the dominant explanation for the gap between Voronoi geometry (nb47) and the empirical composition table (nb46).

---

### F154 — Zscore nonlinearity pulls DCO pairs toward declining_osc centroid (REFUTED)

**Prediction:** All 25 DCO-output pairs: actual features closer to DCO centroid than midpoint (100%).

**Result (Part D):**
- All 64 pairs closer to DCO than midpoint: **24/64 = 37.5%** — refutes "globally pulled toward DCO".
- All 64 pairs **farther from OSC** than midpoint: **54/64 = 84.4%** — clear systematic effect.
- DCO-output pairs closer to DCO than midpoint: **16/25 = 64.0%** (not 100%).
- Distance reduction to DCO for DCO pairs: mean 0.007, median 0.046, max 0.43.

**Verdict:** Refuted as stated. The nonlinearity does **not** uniformly pull mixtures toward declining_osc. The real systematic effect is **departure from oscillator** (84%). DCO classifications win because oscillator gets pushed *out of its own basin*, not because DCO actively attracts.

---

### F155 — Emergent: zscore mixing depletes the oscillator Voronoi basin (84.4%)

**Discovery (Part D):** 54 of 64 pairs land farther from the oscillator centroid after the zscore-after-mix step than their centroid midpoint predicts. Combined with nb47's finding that oscillator owns 77.5% of the Voronoi volume, this is the missing piece: the geometry says oscillator wins, but the nonlinearity systematically *evacuates* the oscillator basin. Mixed signals lose the clean sinusoidal shape that the oscillator centroid encodes; the residual lands in whichever basin best matches the irregular post-mix waveform — most often declining_osc, irregular_osc, or seasonal.

This reframes nb47's "grand centroid paradox" (F149) as a directed flow: **OSC is the geometric attractor under linear interpolation; the zscore nonlinearity is a current that flows away from OSC.**

---

### F156 — Emergent: linear-functional features track theory; shape features do not

**Discovery (Part B):** Two clusters by ρ(actual, midpoint):
- **Linear cluster (ρ > 0.98):** slope, baseline_delta. Both are linear functionals of the signal (regression slope, end-minus-start mean). Mixing then zscoring scales them predictably by ≈√2.
- **Nonlinear cluster (ρ < 0.35):** skewness, kurtosis, lag1_autocorr, zero_crossings. All nonlinear functionals (third/fourth moments, autocorrelation, sign-change count). Their value in the mixture is governed by joint dynamics of A and B, not the average of their individual values.

This explains why the centroid-midpoint model in nb47 works at all (45%): it gets slope/BD right, which carry enough discriminative signal. The four nonlinear features carry the rest, and they require simulation, not interpolation.

---

### F157 — Emergent: kurtosis, ZC, and lag1 carry feature-specific intercept bias

**Discovery (Part A):** Mean deviation (actual − midpoint) is large and *one-sided* for three features:
- kurtosis: **+0.63** (heavier tails after mixing — mixed signals are more leptokurtic than either constituent).
- zero_crossings: **+0.74** (more sign-changes than the average of constituents).
- lag1_autocorr: **−0.52** (less autocorrelated than the average of constituents).

These are not just scale changes — they are constant additive shifts. A linear-correction model `actual ≈ a·midpoint + b` with non-zero `b` would do far better than pure scaling. Mechanistically: mixing a smooth (high lag-1) and a noisy (low lag-1) signal produces a result with intermediate-to-low autocorrelation, but the higher-order tail behaviour and zero-crossing count grow because the two waveforms interfere constructively/destructively in different windows.

---

### Findings
F151–F157 added. Total findings: **157**.

**Summary across nb46 → nb47 → nb48 thread:**
- **nb46:** declining_osc dominates the empirical composition table (43%); no grokking.
- **nb47:** declining_osc is **not** the geometric attractor — oscillator is (77.5% Voronoi). Centroid interpolation predicts only 45%.
- **nb48:** the zscore-after-mix nonlinearity (a) compresses higher moments (skew/kurt), (b) keeps slope/BD near-linear, (c) decorrelates lag1/ZC, (d) systematically *expels* mixtures from the oscillator basin (84%). With actual mean features, prediction accuracy is 96.9%.

The composition attractor is not a property of feature-space geometry — it is a property of the **functional form of feature extraction applied to the mixture**.
